# 01 · Data collection — large-cap equity funds

Both feeds are pulled for all schemes (the downloads don't filter by category); the large-cap slice is taken in `02_cleaning` using the SEBI category segment of the NSDL scheme code.

Two feeds from AMFI, both pulled by code and saved untouched to `raw/`, plus one hand-downloaded summary file used only for AUM.

| Source | URL | Format | Collected |
|---|---|---|---|
| AMFI NAV history | https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx?frmdt=&todt= | semicolon-separated text, no key | 2026-10-24 |
| AMFI Total Expense Ratio | https://www.amfiindia.com/api/populate-te-rdata-revised?MF_ID=All&Month=MM-YYYY&strCat=-1&strType=-1 | monthly, via the `amfipy` wrapper (Excel export of the same dataset) | 2026-10-24 |
| AMFI Fund Performance (AUM) | https://www.amfiindia.com/otherdata/fund-performance — Open Ended · Equity · Large Cap Fund | xlsx, manual download | 2026-10-24 |

**Outputs** — one raw file per source-month in `raw/`, manifests `raw/manifest_nav.csv` and `raw/manifest_ter.csv`, and four interim parquets in `interim/`.

In [1]:
from pathlib import Path
import json, time, io, glob, calendar, re
from datetime import datetime, date
import requests
import pandas as pd
from dateutil.relativedelta import relativedelta

ROOT = Path("..").resolve()
RAW = ROOT / "raw"
INTERIM = ROOT / "interim"
CLEAN = ROOT / "clean"
LOGS = ROOT / "logs"

for p in (RAW, INTERIM, CLEAN, LOGS):
    p.mkdir(parents=True, exist_ok=True)
    print(p, "exists:", p.exists())

TODAY = date.today().isoformat()
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:156.0) Gecko/20100101 Firefox/156.0"}
print("collection date:", TODAY)

C:\Users\da25b048\Documents\madras\da2402 DCV\dcvproject\raw exists: True
C:\Users\da25b048\Documents\madras\da2402 DCV\dcvproject\interim exists: True
C:\Users\da25b048\Documents\madras\da2402 DCV\dcvproject\clean exists: True
C:\Users\da25b048\Documents\madras\da2402 DCV\dcvproject\logs exists: True
collection date: 2026-11-03


In [2]:
# robots.txt and terms of use for every site we scrape
for host in ["https://portal.amfiindia.com", "https://www.amfiindia.com"]:
    r_ = requests.get(f"{host}/robots.txt", headers=HEADERS, timeout=30)
    body = r_.text.strip()
    is_robots = r_.status_code == 200 and "user-agent" in body.lower()
    print(f"=== {host}/robots.txt  (HTTP {r_.status_code}, {'robots file' if is_robots else 'no robots file / HTML returned'})")
    if is_robots:
        print(body[:1500])
    print()

tou = requests.get("https://www.amfiindia.com/terms-of-use", headers=HEADERS, timeout=30)
print("=== terms of use: HTTP", tou.status_code, "|", len(tou.text), "chars")
for phrase in ["personal", "non-commercial", "store electronically", "accuracy", "reverse engineering"]:
    i = tou.text.lower().find(phrase)
    if i >= 0:
        snippet = re.sub(r"<[^>]+>|\s+", " ", tou.text[max(0, i-150): i+200])
        print(f"\n[{phrase}] …{snippet.strip()}…")

=== https://portal.amfiindia.com/robots.txt  (HTTP 404, no robots file / HTML returned)

=== https://www.amfiindia.com/robots.txt  (HTTP 200, robots file)
User-agent: *
Disallow: /admin/
Disallow: /login/
Disallow: /search/

User-agent: Screaming Frog SEO Spider
Allow: /

Sitemap: https://www.amfiindia.com/sitemap.xml

=== terms of use: HTTP 200 | 185911 chars

[personal] …-body1 css-1woj8vn"> AMFI grants you a non-exclusive, personal, non-transferable, non-sublicensable, limited and revocable right to access, use and display this Site on any computers or other electronic display device of which you are a user, for your pe…

[non-commercial] …vocable right to access, use and display this Site on any computers or other electronic display device of which you are a user, for your personal and non-commercial use only (the &quot;Permitted Uses&quot;). You agree to use the Site only for lawful purposes, for the Permitted Uses, and not for the Prohibited Uses. No other use of the Site is auth

### What the checks say (verified 2026-10-24, re-checked 2026-11-03)

- `www.amfiindia.com/robots.txt` disallows only `/admin/`, `/login/` and `/search/` for all user-agents. Neither the TER API nor the terms page is disallowed.
- `portal.amfiindia.com` has no robots.txt (the request returns an error page), so no path is disallowed on the NAV portal.
- AMFI terms of use (https://www.amfiindia.com/terms-of-use): access is licensed for *"personal and non-commercial use only"*; AMFI does not *"guarantee the timeliness, accuracy, reliability, completeness, or usefulness"* of the data; and *"You shall not store electronically any significant portion of any part of the Site."*

 **The three questions from the ethics lecture.** *May I access it?* Yes: neither robots.txt disallows the paths we use, and no login or bot wall is bypassed. *May I copy it?* For analysis, yes: the terms license personal, non-commercial use, which this course project is; the clause against storing "any significant portion" of the Site is read as covering the site, not two public datasets it serves, and we keep only those. *May I keep and use it?* The data stays in the repository for the project and is not redistributed; AMFI's accuracy disclaimer is why `02_cleaning` treats NAV jumps and TER placeholders as possible errors rather than facts. Requests are one per month with a 1.5 s pause.

In [4]:
from amfipy import AMFIClient
client = AMFIClient()

## 1 · NAV history — probe one request

One small request (3 days) to see the format before pulling 45 months.

In [5]:
NAV_URL = "https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx"
params = {"frmdt": "01-Sep-2026", "todt": "03-Sep-2026"}

r = requests.get(NAV_URL, params=params, headers=HEADERS, timeout=60)
print("status:", r.status_code)
print("content-type:", r.headers.get("Content-Type"))
print("length:", len(r.text))
print("---- first 300 chars ----")
print(r.text[:300])

status: 200
content-type: text/plain
length: 3477903
---- first 300 chars ----
Scheme Code;NAV Name;Plan;Option;ISIN Div Payout/ISIN Growth;ISIN Div Reinvestment;Net Asset Value;Date

Open Ended Schemes ( Money Market )


Taurus Mutual Fund
139619;Taurus Investor Education Pool - Unclaimed Dividend - Growth;;;;;10.0000;01-Sep-2026
139619;Taurus Investor Education Pool -


In [6]:
nav_probe_file = RAW / f"amfi_nav_probe_{TODAY}.txt"
nav_probe_file.write_text(r.text, encoding="utf-8")

manifest = pd.DataFrame([{
    "file_name": nav_probe_file.name,
    "source_url": r.url,
    "collected_at": datetime.now().isoformat(timespec="seconds"),
    "http_status": r.status_code,
    "bytes": len(r.content),
    "note": "NAV probe: 01-Sep-2026 to 03-Sep-2026",
}])
manifest.to_csv(RAW / "manifest.csv", index=False)
print(manifest.to_string())

                       file_name                                                                                        source_url         collected_at  http_status    bytes                                   note
0  amfi_nav_probe_2026-11-03.txt  https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx?frmdt=01-Sep-2026&todt=03-Sep-2026  2026-11-03T22:26:09          200  3477903  NAV probe: 01-Sep-2026 to 03-Sep-2026


**Raw NAV file — meaning of each column** (semicolon-separated; section header lines carry the category and AMC name and have no NAV)

| Column | dtype as read | Meaning | Kept? |
|---|---|---|---|
| Scheme Code | str | AMFI numeric scheme-plan-option code (not the NSDL code used in TER) | yes |
| NAV Name | str | full scheme name incl. plan and option | yes |
| Plan | str | `Direct Plan` / `Regular Plan` | yes |
| Option | str | `Growth`, `IDCW`… ~300 spelling variants | yes |
| ISIN Div Payout/ISIN Growth | str | ISIN for growth or payout option | no |
| ISIN Div Reinvestment | str | ISIN for reinvestment option | no |
| Net Asset Value | float | NAV in ₹ per unit | yes |
| Date | str | `DD-Mon-YYYY` | yes |

In [7]:
nav_probe = pd.read_csv(nav_probe_file, sep=";")
print("shape:", nav_probe.shape)
print("dtypes:\n", nav_probe.dtypes)
print("columns:", nav_probe.columns.tolist())
print(nav_probe.head(3).to_string())

real = nav_probe.dropna(subset=["Net Asset Value"]).copy()
print("\nreal rows:", len(real), "of", len(nav_probe))
print("\nPlan value_counts:\n", real["Plan"].value_counts(dropna=False))
print("\nOption distinct values:", real["Option"].nunique())
print("\nScheme Code unique:", nav_probe["Scheme Code"].nunique())
print("rows-per-code distribution:")
print(real.groupby("Scheme Code").size().describe())

shape: (27309, 8)
dtypes:
 Scheme Code                        str
NAV Name                           str
Plan                               str
Option                             str
ISIN Div Payout/ISIN Growth        str
ISIN Div Reinvestment              str
Net Asset Value                float64
Date                               str
dtype: object
columns: ['Scheme Code', 'NAV Name', 'Plan', 'Option', 'ISIN Div Payout/ISIN Growth', 'ISIN Div Reinvestment', 'Net Asset Value', 'Date']
                           Scheme Code                                                      NAV Name Plan Option ISIN Div Payout/ISIN Growth ISIN Div Reinvestment  Net Asset Value         Date
0  Open Ended Schemes ( Money Market )                                                           NaN  NaN    NaN                         NaN                   NaN              NaN          NaN
1                   Taurus Mutual Fund                                                           NaN  NaN    NaN           

## 2 · TER — probe one month

The brief's JSON endpoint is paginated; the `amfipy` wrapper requests AMFI's Excel export of the same monthly table, which gives all schemes in one call. One row per scheme per *TER date* — a scheme repeats within a month whenever its TER changed.

In [8]:
excel_bytes = client.ter.download_excel(month="08-2026")
print("bytes:", len(excel_bytes))

ter_probe = pd.read_excel(io.BytesIO(excel_bytes))
print("shape:", ter_probe.shape)
print("columns:", ter_probe.columns.tolist())
print("unique NSDL Scheme Code:", ter_probe["NSDL Scheme Code"].nunique())
print("TER Date min/max:", ter_probe["TER Date"].min(), "->", ter_probe["TER Date"].max())

lc_mask = ter_probe["Scheme Category"].str.contains("Large Cap", case=False, na=False)
print("\nLarge Cap rows:", lc_mask.sum())
print("Large Cap unique schemes:", ter_probe.loc[lc_mask, "NSDL Scheme Code"].nunique())
print("\nlarge-cap label variants:")
print(ter_probe.loc[lc_mask, "Scheme Category"].value_counts().to_string())

# save deduped probe as CSV
ter_probe_file = RAW / f"amfi_ter_probe_2026-08_{TODAY}.csv"
(ter_probe.sort_values("TER Date")
          .drop_duplicates(subset=["NSDL Scheme Code"], keep="first")
          .to_csv(ter_probe_file, index=False))

manifest = pd.read_csv(RAW / "manifest.csv")
manifest = pd.concat([manifest, pd.DataFrame([{
    "file_name": ter_probe_file.name,
    "source_url": "amfipy client.ter.download_excel(month=08-2026) -> csv",
    "collected_at": datetime.now().isoformat(timespec="seconds"),
    "http_status": 200,
    "bytes": ter_probe_file.stat().st_size,
    "note": f"TER probe: 2026-08, {ter_probe_file.stat().st_size:,} bytes",
}])], ignore_index=True)
manifest.to_csv(RAW / "manifest.csv", index=False)
print("\nmanifest now has", len(manifest), "rows")

bytes: 4475976
shape: (63699, 15)
columns: ['NSDL Scheme Code', 'Scheme Name', 'Scheme Type', 'Scheme Category', 'TER Date', 'Regular Plan - Base Expense Ratio (BER) (%)', 'Regular Plan - Brokerage cost (%)', 'Regular Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Regular Plan - Statutory Levies (including GST) (%)', 'Regular Plan - Total TER (%)', 'Direct Plan - Base Expense Ratio (BER) (%)', 'Direct Plan - Brokerage cost (%)', 'Direct Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Direct Plan - Statutory Levies (including GST) (%)', 'Direct Plan - Total TER (%)']
unique NSDL Scheme Code: 2150
TER Date min/max: 2026-08-01 00:00:00 -> 2026-08-31 00:00:00

Large Cap rows: 1055
Large Cap unique schemes: 35

large-cap label variants:
Scheme Category
Equity Scheme - Large Cap Fund     724
Equity Schemes - Large Cap Fund    331

manifest now has 2 rows


In [9]:
START = date(2023, 1, 1)
END   = date(2026, 9, 30)

def month_list(start, end):
    cur = date(start.year, start.month, 1)
    while cur <= end:
        yield cur.year, cur.month
        cur = cur + relativedelta(months=1)

months = list(month_list(START, END))
print("months:", len(months), "|", months[0], "to", months[-1])

def get_with_retry(url, params, headers, timeout=120, tries=3, sleep=2.0):
    for k in range(tries):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp
            time.sleep(sleep * (k + 1))
        except requests.exceptions.RequestException:
            time.sleep(sleep * (k + 1))
    return None

months: 45 | (2023, 1) to (2026, 9)


## 3 · Bulk pull: Jan 2023 → Sep 2026

Both loops skip a month if its file already exists, so the notebook re-runs without re-downloading. 1.5 s pause between requests.

In [10]:
nav_manifest = []

for year, month in months:
    last_day = calendar.monthrange(year, month)[1]
    frm = date(year, month, 1).strftime("%d-%b-%Y")
    to  = date(year, month, last_day).strftime("%d-%b-%Y")
    fname = RAW / f"amfi_nav_{year}-{month:02d}.txt"

    if fname.exists():
        print(f"skip {fname.name} ({fname.stat().st_size:,} bytes)")
        nav_manifest.append({"file_name": fname.name, "source_url": NAV_URL,
                             "collected_at": datetime.fromtimestamp(fname.stat().st_mtime).isoformat(timespec="seconds"),
                             "http_status": 200, "bytes": fname.stat().st_size,
                             "note": f"NAV {frm} to {to} (cached)"})
        continue

    resp = get_with_retry(NAV_URL, {"frmdt": frm, "todt": to}, HEADERS)
    if resp is None:
        print(f"FAILED {year}-{month:02d}")
        nav_manifest.append({"file_name": None, "source_url": NAV_URL,
                             "collected_at": datetime.now().isoformat(timespec="seconds"),
                             "http_status": "FAILED", "bytes": 0, "note": f"NAV {frm} to {to}"})
        continue

    fname.write_text(resp.text, encoding="utf-8")
    print(f"wrote {fname.name}  {len(resp.content):>9,} bytes")
    nav_manifest.append({"file_name": fname.name, "source_url": resp.url,
                         "collected_at": datetime.now().isoformat(timespec="seconds"),
                         "http_status": resp.status_code, "bytes": len(resp.content),
                         "note": f"NAV {frm} to {to}"})
    time.sleep(1.5)

pd.DataFrame(nav_manifest).to_csv(RAW / "manifest_nav.csv", index=False)
print("\nNAV written:", sum(1 for x in nav_manifest if x["file_name"]),
      "| failed:", sum(1 for x in nav_manifest if x["file_name"] is None))

skip amfi_nav_2023-01.txt (220,513 bytes)
skip amfi_nav_2023-02.txt (220,513 bytes)
skip amfi_nav_2023-03.txt (220,513 bytes)
skip amfi_nav_2023-04.txt (17,946,652 bytes)
skip amfi_nav_2023-05.txt (21,840,394 bytes)
skip amfi_nav_2023-06.txt (21,214,378 bytes)
skip amfi_nav_2023-07.txt (21,325,073 bytes)
skip amfi_nav_2023-08.txt (21,779,987 bytes)
skip amfi_nav_2023-09.txt (19,570,974 bytes)
skip amfi_nav_2023-10.txt (20,500,929 bytes)
skip amfi_nav_2023-11.txt (20,438,258 bytes)
skip amfi_nav_2023-12.txt (20,637,123 bytes)
skip amfi_nav_2024-01.txt (21,608,899 bytes)
skip amfi_nav_2024-02.txt (21,174,978 bytes)
skip amfi_nav_2024-03.txt (20,155,232 bytes)
skip amfi_nav_2024-04.txt (20,224,513 bytes)
skip amfi_nav_2024-05.txt (21,718,354 bytes)
skip amfi_nav_2024-06.txt (20,499,460 bytes)
skip amfi_nav_2024-07.txt (23,160,878 bytes)
skip amfi_nav_2024-08.txt (22,525,064 bytes)
skip amfi_nav_2024-09.txt (21,997,071 bytes)
skip amfi_nav_2024-10.txt (23,628,089 bytes)
skip amfi_nav_2024-

## 4 · TER bulk pull

In [11]:
ter_manifest = []

for year, month in months:
    month_str = f"{month:02d}-{year}"
    fname = RAW / f"amfi_ter_{year}-{month:02d}.csv"

    if fname.exists():
        print(f"skip {fname.name} ({fname.stat().st_size:,} bytes)")
        ter_manifest.append({"file_name": fname.name,
                             "source_url": "amfipy client.ter.download_excel",
                             "collected_at": datetime.fromtimestamp(fname.stat().st_mtime).isoformat(timespec="seconds"),
                             "http_status": 200, "bytes": fname.stat().st_size,
                             "note": f"TER {month_str} (cached)"})
        continue

    try:
        xb = client.ter.download_excel(month=month_str)
        df_m = pd.read_excel(io.BytesIO(xb))
        df_m = df_m.sort_values("TER Date").drop_duplicates(
            subset=["NSDL Scheme Code"], keep="first")
        df_m.to_csv(fname, index=False)
        print(f"wrote {fname.name}  {len(df_m):>5,} schemes  {fname.stat().st_size:>8,} bytes")
        ter_manifest.append({"file_name": fname.name,
                             "source_url": "amfipy client.ter.download_excel",
                             "collected_at": datetime.now().isoformat(timespec="seconds"),
                             "http_status": 200, "bytes": fname.stat().st_size,
                             "note": f"TER {month_str}, {len(df_m)} unique schemes"})
    except Exception as e:
        print(f"FAILED {month_str}: {type(e).__name__}: {e}")
        ter_manifest.append({"file_name": None, "source_url": "amfipy",
                             "collected_at": datetime.now().isoformat(timespec="seconds"),
                             "http_status": "FAILED", "bytes": 0, "note": f"TER {month_str}: {e}"})

    time.sleep(1.5)

pd.DataFrame(ter_manifest).to_csv(RAW / "manifest_ter.csv", index=False)
print("\nTER written:", sum(1 for x in ter_manifest if x["file_name"]),
      "| failed:", sum(1 for x in ter_manifest if x["file_name"] is None))

skip amfi_ter_2023-01.csv (2,579 bytes)
skip amfi_ter_2023-02.csv (2,588 bytes)
skip amfi_ter_2023-03.csv (2,588 bytes)
skip amfi_ter_2023-04.csv (2,549 bytes)
skip amfi_ter_2023-05.csv (2,584 bytes)
skip amfi_ter_2023-06.csv (2,617 bytes)
skip amfi_ter_2023-07.csv (15,898 bytes)
skip amfi_ter_2023-08.csv (16,629 bytes)
skip amfi_ter_2023-09.csv (2,673 bytes)
skip amfi_ter_2023-10.csv (2,692 bytes)
skip amfi_ter_2023-11.csv (2,692 bytes)
skip amfi_ter_2023-12.csv (2,692 bytes)
skip amfi_ter_2024-01.csv (2,692 bytes)
skip amfi_ter_2024-02.csv (2,692 bytes)
skip amfi_ter_2024-03.csv (2,692 bytes)
skip amfi_ter_2024-04.csv (2,691 bytes)
skip amfi_ter_2024-05.csv (2,692 bytes)
skip amfi_ter_2024-06.csv (2,853 bytes)
skip amfi_ter_2024-07.csv (2,996 bytes)
skip amfi_ter_2024-08.csv (2,691 bytes)
skip amfi_ter_2024-09.csv (15,658 bytes)
skip amfi_ter_2024-10.csv (289,514 bytes)
skip amfi_ter_2024-11.csv (284,697 bytes)
skip amfi_ter_2024-12.csv (291,644 bytes)
skip amfi_ter_2025-01.csv (292,

In [12]:
nav_frames = []
nav_errors = []
USECOLS = ["Scheme Code", "NAV Name", "Plan", "Option", "Net Asset Value", "Date"]

for f in sorted(glob.glob(str(RAW / "amfi_nav_*.txt"))):
    name = Path(f).name
    if "probe" in name or name < "amfi_nav_2024-10.txt":   # usable window only
        continue
    try:
        d = pd.read_csv(f, sep=";", on_bad_lines="skip", low_memory=False, usecols=USECOLS)
        d = d.dropna(subset=["Net Asset Value"])            # drops section-header lines
        d["_source_file"] = name
        nav_frames.append(d)
        print(f"ok    {name}  {d.shape}")
    except Exception as e:
        nav_errors.append((name, type(e).__name__, str(e)[:120]))
        print(f"FAIL  {name}  {type(e).__name__}: {str(e)[:120]}")

nav_usable = pd.concat(nav_frames, ignore_index=True)
del nav_frames
print("\nNAV usable shape:", nav_usable.shape, "| files:", nav_usable["_source_file"].nunique())

ok    amfi_nav_2024-10.txt  (176715, 7)
ok    amfi_nav_2024-11.txt  (149981, 7)
ok    amfi_nav_2024-12.txt  (172715, 7)
ok    amfi_nav_2025-01.txt  (188345, 7)
ok    amfi_nav_2025-02.txt  (155716, 7)
ok    amfi_nav_2025-03.txt  (169618, 7)
ok    amfi_nav_2025-04.txt  (158611, 7)
ok    amfi_nav_2025-05.txt  (175060, 7)
ok    amfi_nav_2025-06.txt  (177881, 7)
ok    amfi_nav_2025-07.txt  (194383, 7)
ok    amfi_nav_2025-08.txt  (167181, 7)
ok    amfi_nav_2025-09.txt  (186251, 7)
ok    amfi_nav_2025-10.txt  (175609, 7)
ok    amfi_nav_2025-11.txt  (168961, 7)
ok    amfi_nav_2025-12.txt  (193347, 7)
ok    amfi_nav_2026-01.txt  (180017, 7)
ok    amfi_nav_2026-02.txt  (175975, 7)
ok    amfi_nav_2026-03.txt  (178791, 7)
ok    amfi_nav_2026-04.txt  (176470, 7)
ok    amfi_nav_2026-05.txt  (171883, 7)
ok    amfi_nav_2026-06.txt  (185905, 7)
ok    amfi_nav_2026-07.txt  (202378, 7)
ok    amfi_nav_2026-08.txt  (185003, 7)
ok    amfi_nav_2026-09.txt  (188822, 7)

NAV usable shape: (4255618, 7) | files:

In [13]:
small_navs = [f for f in sorted(glob.glob(str(RAW / "amfi_nav_*.txt")))
              if "probe" not in f and Path(f).stat().st_size < 1_000_000]
print("small NAV files (< 1 MB):")
for f in small_navs:
    print(f"  {Path(f).name}  {Path(f).stat().st_size:,} bytes")
    head = Path(f).read_text(encoding="utf-8", errors="ignore")[:200]
    print(f"    first 200 chars: {head!r}")

small NAV files (< 1 MB):
  amfi_nav_2023-01.txt  220,513 bytes
    first 200 chars: '<!DOCTYPE html><html lang="en" data-theme="light"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="stylesheet" href="/_next/static/css/d0eed'
  amfi_nav_2023-02.txt  220,513 bytes
    first 200 chars: '<!DOCTYPE html><html lang="en" data-theme="light"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="stylesheet" href="/_next/static/css/d0eed'
  amfi_nav_2023-03.txt  220,513 bytes
    first 200 chars: '<!DOCTYPE html><html lang="en" data-theme="light"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="stylesheet" href="/_next/static/css/d0eed'


In [14]:
def read_ter_csv(path):
    d = pd.read_csv(path, on_bad_lines="skip")
    d = d[d["NSDL Scheme Code"].notna()].copy()
    d["TER Date"] = pd.to_datetime(d["TER Date"], errors="coerce")
    d = d[d["TER Date"].notna()].copy()
    d["_source_file"] = Path(path).name
    m = re.search(r"amfi_ter_(\d{4})-(\d{2})", Path(path).name)
    if m:
        d["Month"] = f"{m.group(2)}-{m.group(1)}"
    return d

ter_frames = []
ter_dropped = []

for f in sorted(glob.glob(str(RAW / "amfi_ter_*.csv"))):
    if "probe" in f:
        continue
    raw_n = sum(1 for _ in open(f, encoding="utf-8")) - 1
    d = read_ter_csv(f)
    kept = len(d)
    dropped = max(0, raw_n - kept)
    if dropped:
        ter_dropped.append((Path(f).name, raw_n, kept, dropped))
    ter_frames.append(d)

ter_all = pd.concat(ter_frames, ignore_index=True)

# drop trailing unnamed columns created by on_bad_lines="skip" reading disclaimer rows
ter_all = ter_all.loc[:, [c for c in ter_all.columns if not str(c).startswith("Unnamed")]]

# proper month ordering
ter_all["MonthPeriod"] = (pd.to_datetime(ter_all["Month"], format="%m-%Y").dt.to_period("M"))

print("TER combined shape:", ter_all.shape)
print("columns:", ter_all.columns.tolist())
print("unique NSDL Scheme Code:", ter_all["NSDL Scheme Code"].nunique())
print("unique Month:", ter_all["Month"].nunique())
print("rows dropped as disclaimer/footer:", sum(x[3] for x in ter_dropped))

print("\nTER schemes per month (chronological):")
per_month = (ter_all.groupby("MonthPeriod")["NSDL Scheme Code"]
                    .nunique()
                    .rename_axis("Month")
                    .reset_index())
per_month["Month"] = per_month["Month"].dt.strftime("%m-%Y")
print(per_month.to_string(index=False))

TER combined shape: (47512, 26)
columns: ['NSDL Scheme Code', 'Scheme Name', 'Scheme Type', 'Scheme Category', 'TER Date', 'Regular Plan - Base TER (%)', 'Regular Plan - Additional expense as per Regulation 52(6A)(b) (%)', 'Regular Plan - Additional expense as per Regulation 52(6A)(c) (%)', 'Regular Plan - GST (%)', 'Regular Plan - Total TER (%)', 'Direct Plan - Base TER (%)', 'Direct Plan - Additional expense as per Regulation 52(6A)(b) (%)', 'Direct Plan - Additional expense as per Regulation 52(6A)(c) (%)', 'Direct Plan - GST (%)', 'Direct Plan - Total TER (%)', '_source_file', 'Month', 'Regular Plan - Base Expense Ratio (BER) (%)', 'Regular Plan - Brokerage cost (%)', 'Regular Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Regular Plan - Statutory Levies (including GST) (%)', 'Direct Plan - Base Expense Ratio (BER) (%)', 'Direct Plan - Brokerage cost (%)', 'Direct Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Direct Pla

**Why the usable window starts 2024-10.** The table above shows the TER export is empty or near-empty for every month before 2024-10 (a few dozen schemes), and full coverage (~1,800 to 2,100 schemes per month) begins in 2024-10. The brief says the API is monthly from 2019, but the export AMFI serves for earlier months does not carry the full table. The analysis is therefore restricted to 2024-10 to 2026-09, 24 months. The NAV loader in the previous cell reads only those months for the same reason. This is the main limitation of the project and is repeated in the report.

## 6 · Window selection and interim parquets

Parquet is used for `interim/` because the NAV frame is 4.3 M rows: Parquet keeps dtypes (CSV would re-read `Date` and `Net Asset Value` as strings), is ~10× smaller than the text files, and reads in seconds. Dtypes are checked after read-back in the last cell.

In [16]:
nav_usable["Date"] = pd.to_datetime(nav_usable["Date"], format="%d-%b-%Y", errors="coerce")
nav_usable["Net Asset Value"] = pd.to_numeric(nav_usable["Net Asset Value"], errors="coerce")
nav_usable["MonthStr"] = nav_usable["Date"].dt.to_period("M").astype(str)

ter_usable = ter_all[ter_all["MonthPeriod"] >= pd.Period("2024-10", freq="M")].copy()
ter_usable["MonthStr"] = ter_usable["MonthPeriod"].astype(str)
ter_usable = ter_usable.drop(columns=["MonthPeriod"])

print("usable window: 2024-10 to 2026-09")
print("NAV usable:", nav_usable.shape, "| months:", nav_usable["MonthStr"].nunique())
print("TER usable:", ter_usable.shape, "| months:", ter_usable["MonthStr"].nunique())
print("\nTER schemes per month:")
print(ter_usable.groupby("MonthStr")["NSDL Scheme Code"].nunique().to_string())

nav_usable.to_parquet(INTERIM / "nav_usable_2024-10_2026-09.parquet", index=False)
ter_usable.to_parquet(INTERIM / "ter_usable_2024-10_2026-09.parquet", index=False)
print("\nwrote nav_usable:", len(nav_usable), "rows | ter_usable:", len(ter_usable), "rows")

usable window: 2024-10 to 2026-09
NAV usable: (4255618, 8) | months: 24
TER usable: (47228, 26) | months: 24

TER schemes per month:
MonthStr
2024-10    1782
2024-11    1749
2024-12    1794
2025-01    1797
2025-02    1822
2025-03    1857
2025-04    1860
2025-05    1873
2025-06    1893
2025-07    1926
2025-08    1952
2025-09    1966
2025-10    1984
2025-11    2005
2025-12    2029
2026-01    2044
2026-02    2070
2026-03    2101
2026-04    2119
2026-05    2098
2026-06    2094
2026-07    2113
2026-08    2138
2026-09    2162

wrote nav_usable: 4255618 rows | ter_usable: 47228 rows


In [17]:
nav_check = pd.read_parquet(INTERIM / "nav_usable_2024-10_2026-09.parquet")
ter_check = pd.read_parquet(INTERIM / "ter_usable_2024-10_2026-09.parquet")
print("nav_usable:", nav_check.shape)
print(nav_check.dtypes)
print("\nter_usable:", ter_check.shape)
print(ter_check.dtypes)

nav_usable: (4255618, 8)
Scheme Code                   str
NAV Name                      str
Plan                          str
Option                        str
Net Asset Value           float64
Date               datetime64[us]
_source_file                  str
MonthStr                      str
dtype: object

ter_usable: (47228, 26)
NSDL Scheme Code                                                                                 str
Scheme Name                                                                                      str
Scheme Type                                                                                      str
Scheme Category                                                                                  str
TER Date                                                                              datetime64[us]
Regular Plan - Base TER (%)                                                                  float64
Regular Plan - Additional expense as per Regulation 52(6A)

## 7 · Hand-downloaded file: AMFI Fund Performance (AUM)

Used only for the AUM shown on hover in the dashboard. Downloaded by hand on 2026-10-24 from https://www.amfiindia.com/otherdata/fund-performance with the page filters **Scheme Type = Open Ended · Scheme Category = Equity · Sub-category = Large Cap Fund**, then the page's Excel download. The sheet header reads "Crisil Intelligence — Generated on 24-Oct-2026 10:44". Saved as `raw/amfi_fund_performance_largecap_2026-10-24.xlsx`, unedited.

In [18]:
aum_file = RAW / "amfi_fund_performance_largecap_2026-10-24.xlsx"
aum_raw = pd.read_excel(aum_file, sheet_name="Fund_Performance", header=4)
print("shape:", aum_raw.shape)
print(aum_raw.dtypes.to_string())
print(aum_raw[["Scheme Name", "NAV Date", "NAV Regular", "NAV Direct", "Daily AUM (Cr.)"]].head(5).to_string())

shape: (40, 32)
Scheme Name                                        str
Benchmark                                          str
Riskometer Scheme                                  str
Riskometer Benchmark                               str
NAV Date                                datetime64[us]
NAV Regular                                    float64
NAV Direct                                     float64
Return 1 Year (%) Regular                      float64
Return 1 Year (%) Direct                       float64
Return 1 Year (%) Benchmark                    float64
Information Ratio* 1 Year (Regular)            float64
Information Ratio* 1 Year (Direct)             float64
Return 3 Year (%) Regular                      float64
Return 3 Year (%) Direct                       float64
Return 3 Year (%) Benchmark                    float64
Information Ratio* 3 Year (Regular)            float64
Information Ratio* 3 Year (Direct)             float64
Return 5 Year (%) Regular                      fl